# Modul 06 · Notebook 03 — RAG di Stack NVIDIA

Di Modul 05 kamu membangun **RAG**. Di sini kita jalankan di **stack NVIDIA**: embedding ringan di **CPU**, generator di **cloud (NIM)** — tanpa model berat di GPU-mu.

Ini juga **memperbaiki** notebook lama yang men-generate dengan **GPT-2** (kecil & usang, jawabannya kacau). Generator-nya kita ganti dengan **`nvidia/nemotron-3-nano-30b-a3b`** via NIM.

> 🔑 Pakai `NVIDIA_API_KEY` yang sama (Colab Secrets) seperti nb02.

In [ ]:
# Instal + ambil helper bersama
!pip -q install openai sentence-transformers faiss-cpu
import os
if not os.path.exists("nvidia_utils.py"):
    !wget -q https://raw.githubusercontent.com/chmdznr/navasena-gen-ml-course/module06-rework/06_nvidia_ecosystem/tools/nvidia_utils.py

In [ ]:
# Key dari Colab Secrets
import os
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
print("Key termuat:", bool(os.environ.get("NVIDIA_API_KEY")))

In [ ]:
# 1) RETRIEVE: korpus kecil -> embedding (multilingual MiniLM, di CPU) -> indeks FAISS
from sentence_transformers import SentenceTransformer
import faiss

emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu")
DOCS = [
    "GPU NVIDIA memiliki ribuan core yang bekerja paralel, ideal untuk perkalian matriks pada AI.",
    "NVIDIA NIM menyajikan model besar lewat API kompatibel-OpenAI di integrate.api.nvidia.com.",
    "Quantization 4-bit memangkas memori model secara drastis sehingga model besar muat di GPU kecil.",
    "Tensor Core mempercepat komputasi FP16 di GPU NVIDIA secara signifikan.",
    "RAG menggabungkan pencarian dokumen dengan LLM agar jawaban berdasarkan sumber yang relevan.",
    "CUDA adalah platform NVIDIA untuk memprogram GPU bagi komputasi umum, bukan hanya grafis.",
]
X = emb.encode(DOCS, normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(X.shape[1]); index.add(X)
print("Jumlah dokumen terindeks:", index.ntotal)

In [ ]:
# 2) Cari dokumen paling relevan untuk sebuah pertanyaan
q = "Bagaimana cara menjalankan model besar di GPU yang kecil?"
qv = emb.encode([q], normalize_embeddings=True).astype("float32")
D, I = index.search(qv, k=3)
konteks = "\n".join(f"[{i+1}] {DOCS[i]}" for i in I[0])
print("Konteks terambil:\n", konteks)

In [ ]:
# 3) GENERATE via NIM (Nemotron 3 Nano) -- jawaban grounded + sebut nomor sumber
from nvidia_utils import nim_client, nim_chat
client = nim_client()
prompt = ("Gunakan HANYA konteks berikut untuk menjawab, dan sebutkan nomor sumber [n].\n"
          f"Konteks:\n{konteks}\n\nPertanyaan: {q}")
print(nim_chat(client, "nvidia/nemotron-3-nano-30b-a3b",
               messages=[{"role": "user", "content": prompt}]))

## Yang baru saja terjadi

1. **Retrieve** — embedding ringan (CPU) menemukan dokumen relevan via FAISS.
2. **Augment** — dokumen disisipkan ke prompt sebagai *konteks*.
3. **Generate** — Nemotron 3 Nano (NIM) menjawab **berdasarkan konteks** + menyebut nomor sumber.

Jawaban menyebut **[n]** (sitasi) dan tidak mengarang. Tapi bagaimana kalau model **tetap** mengarang? Itu masalah **halusinasi** — di **nb07** kita pasang **grounding rail** (`self check facts`) yang membungkus persis pipeline ini.

## 🧪 Latihan

1. Tambahkan 1 dokumen baru ke `DOCS`, lalu ajukan pertanyaan tentangnya — apakah ikut terambil?
2. Ubah `k` dari 3 menjadi 1 — apakah jawaban jadi kurang lengkap?
3. Ajukan pertanyaan yang **tidak ada** di korpus. Apakah model jujur bilang tidak tahu, atau mengarang? (Catat — motivasi nb07.)